# 07 — Dataset Split for Baseline Training

Notebook นี้ต่อจาก `06_final_labeled_lip_dataset_builder.ipynb`

Input:
- `dataset/final_dataset/video001/manifest_train_ready.csv`

Output:
- `dataset/splits/video001/train.csv`
- `dataset/splits/video001/val.csv`
- `dataset/splits/video001/test.csv`
- `dataset/splits/video001/split_summary.csv`

## หลักการ split

เนื่องจากตอนนี้มี **วิดีโอเดียว + ผู้พูดหลัก 2 คน** เราจะใช้ **speaker-aware chronological split** เพื่อ baseline:

- Train 70%
- Validation 15%
- Test 15%

แต่ละ speaker จะถูกเรียงตามเวลา แล้วแบ่งเป็นช่วงต่อเนื่อง เพื่อหลีกเลี่ยงการสุ่ม clip ที่อยู่ติดกันไปคนละ split ซึ่งเสี่ยง data leakage

> ชุดนี้เหมาะสำหรับ baseline experiment เท่านั้น ยังไม่ใช่การวัด generalization กับผู้พูดใหม่

## Cell 1 — Path

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

MANIFEST_CSV = (
    PROJECT_ROOT
    / "dataset"
    / "final_dataset"
    / VIDEO_ID
    / "manifest_train_ready.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "dataset"
    / "splits"
    / VIDEO_ID
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("MANIFEST_CSV:", MANIFEST_CSV)
print("OUTPUT_ROOT :", OUTPUT_ROOT)

if not MANIFEST_CSV.exists():
    raise FileNotFoundError(MANIFEST_CSV)

MANIFEST_CSV: D:\Face_detection\dataset\final_dataset\video001\manifest_train_ready.csv
OUTPUT_ROOT : D:\Face_detection\dataset\splits\video001


## Cell 2 — Load Manifest

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(MANIFEST_CSV)

required_cols = {
    "clip_id",
    "video_id",
    "segment_id",
    "speaker_id",
    "start_time",
    "end_time",
    "duration",
    "start_frame",
    "end_frame",
    "num_frames",
    "text",
    "frame_paths",
}

missing = required_cols - set(df.columns)

if missing:
    raise ValueError(f"Manifest ขาด columns: {missing}")

df["speaker_id"] = df["speaker_id"].astype(int)
df["segment_id"] = df["segment_id"].astype(int)
df["start_time"] = df["start_time"].astype(float)
df["end_time"] = df["end_time"].astype(float)

print("Total clips:", len(df))
print("Speakers   :", sorted(df["speaker_id"].unique().tolist()))

display(df.head())

Total clips: 732
Speakers   : [0, 1]


,clip_id,video_id,segment_id,speaker_id,start_time,end_time,duration,start_frame,end_frame,num_frames,coverage,continuity,text,frame_paths,frame_indices,label_status
0,video001_seg000010_spk000,video001,10,0,50.76,52.78,2.02,1521,1582,62,1.0,1.0,เพราะปฏิเสธไม่ได้เลยว่าการรู้ภาษาอังกฤษนี้,dataset\mouth_crops_final\video001\speaker_000...,1521|1522|1523|1524|1525|1526|1527|1528|1529|1...,weak_label_auto
1,video001_seg000011_spk000,video001,11,0,52.78,55.40,2.62,1581,1661,81,1.0,1.0,ก็จะช่วยเปิดช่องทางเราในหลายๆด้าน,dataset\mouth_crops_final\video001\speaker_000...,1581|1582|1583|1584|1585|1586|1587|1588|1589|1...,weak_label_auto
2,video001_seg000012_spk000,video001,12,0,55.40,56.92,1.52,1660,1706,47,1.0,1.0,อย่างเช่นการเอาไปอับเงินเดือน,dataset\mouth_crops_final\video001\speaker_000...,1660|1661|1662|1663|1664|1665|1666|1667|1668|1...,weak_label_auto
3,video001_seg000013_spk000,video001,13,0,56.92,58.76,1.84,1705,1762,58,1.0,1.0,หรือการเติบโตในหน้าที่การงาน,dataset\mouth_crops_final\video001\speaker_000...,1705|1706|1707|1708|1709|1710|1711|1712|1713|1...,weak_label_auto
4,video001_seg000016_spk001,video001,16,1,67.58,69.54,1.96,2025,2085,61,1.0,1.0,เขาจะเป็นアプลิเคชั่นเรียนภาษาอังกฤษ,dataset\mouth_crops_final\video001\speaker_001...,2025|2026|2027|2028|2029|2030|2031|2032|2033|2...,weak_label_auto


## Cell 3 — ตรวจข้อมูลซ้ำก่อน Split

In [3]:
print("Duplicate clip_id:", df["clip_id"].duplicated().sum())

duplicate_segments = (
    df.duplicated(
        subset=[
            "video_id",
            "segment_id",
            "speaker_id",
        ]
    ).sum()
)

print("Duplicate segment-speaker rows:", duplicate_segments)

if df["clip_id"].duplicated().any():
    raise RuntimeError("พบ clip_id ซ้ำ กรุณาตรวจ manifest ก่อน split")

Duplicate clip_id: 0
Duplicate segment-speaker rows: 0


## Cell 4 — Split Config

ใช้ 70 / 15 / 15

In [4]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

assert abs(
    TRAIN_RATIO
    + VAL_RATIO
    + TEST_RATIO
    - 1.0
) < 1e-9

print("Train:", TRAIN_RATIO)
print("Val  :", VAL_RATIO)
print("Test :", TEST_RATIO)

Train: 0.7
Val  : 0.15
Test : 0.15


## Cell 5 — Speaker-aware Chronological Split

แต่ละ speaker:
1. เรียงตาม `start_time`
2. 70% แรก → train
3. 15% ถัดมา → val
4. 15% ท้าย → test

In [5]:
split_parts = []

for speaker_id, group in df.groupby("speaker_id"):

    group = (
        group
        .sort_values(
            ["start_time", "segment_id"]
        )
        .reset_index(drop=True)
        .copy()
    )

    n = len(group)

    train_end = int(
        np.floor(
            n * TRAIN_RATIO
        )
    )

    val_end = int(
        np.floor(
            n * (
                TRAIN_RATIO
                + VAL_RATIO
            )
        )
    )

    # กันกรณีจำนวนข้อมูลน้อย
    train_end = max(
        1,
        min(
            train_end,
            n
        )
    )

    val_end = max(
        train_end,
        min(
            val_end,
            n
        )
    )

    group["split"] = "test"

    group.loc[
        0:train_end - 1,
        "split"
    ] = "train"

    if val_end > train_end:
        group.loc[
            train_end:val_end - 1,
            "split"
        ] = "val"

    split_parts.append(
        group
    )

split_df = pd.concat(
    split_parts,
    ignore_index=True
)

print(
    split_df["split"]
    .value_counts()
)

display(
    split_df[
        [
            "clip_id",
            "speaker_id",
            "start_time",
            "end_time",
            "duration",
            "split",
            "text",
        ]
    ].head(30)
)

split
train    511
test     111
val      110
Name: count, dtype: int64


,clip_id,speaker_id,start_time,end_time,duration,split,text
0,video001_seg000010_spk000,0,50.76,52.78,2.02,train,เพราะปฏิเสธไม่ได้เลยว่าการรู้ภาษาอังกฤษนี้
1,video001_seg000011_spk000,0,52.78,55.40,2.62,train,ก็จะช่วยเปิดช่องทางเราในหลายๆด้าน
2,video001_seg000012_spk000,0,55.40,56.92,1.52,train,อย่างเช่นการเอาไปอับเงินเดือน
3,video001_seg000013_spk000,0,56.92,58.76,1.84,train,หรือการเติบโตในหน้าที่การงาน
4,video001_seg000056_spk000,0,159.20,160.54,1.34,train,รับเสร็จง่ายๆเลยนะคะ
5,video001_seg000057_spk000,0,160.54,162.66,2.12,train,เพียงแค่กดตรงลิ้งใน Description
6,video001_seg000059_spk000,0,165.12,167.04,1.92,train,หลังจากนั้นให้ใส่ Code TMI
7,video001_seg000060_spk000,0,167.04,170.74,3.70,train,ตอน Checkout ก็สามารถรับส่วนรถสุดพิเศษไปได้เลยค่ะ
8,video001_seg000068_spk000,0,214.31,215.81,1.50,train,คุณในกลุ่มของเรานะคะ
9,video001_seg000072_spk000,0,224.48,226.16,1.68,train,เจ้าของ Post เล่าว่า


## Cell 6 — Split Summary

In [6]:
summary = (
    split_df
    .groupby(
        ["split", "speaker_id"]
    )
    .agg(
        clips=("clip_id", "count"),
        total_duration_sec=("duration", "sum"),
        mean_duration_sec=("duration", "mean"),
        mean_frames=("num_frames", "mean"),
        first_time=("start_time", "min"),
        last_time=("end_time", "max"),
    )
    .reset_index()
)

summary["total_duration_min"] = (
    summary["total_duration_sec"]
    / 60.0
)

display(summary)

print("\nOverall:")
overall = (
    split_df
    .groupby("split")
    .agg(
        clips=("clip_id", "count"),
        total_duration_sec=("duration", "sum"),
        speakers=("speaker_id", "nunique"),
    )
    .reset_index()
)

overall["total_duration_min"] = (
    overall["total_duration_sec"]
    / 60.0
)

display(overall)

,split,speaker_id,clips,total_duration_sec,mean_duration_sec,mean_frames,first_time,last_time,total_duration_min
0,test,0,45,95.69,2.126444,65.600000,4598.23,5905.77,1.594833
1,test,1,66,157.72,2.389697,73.606061,5475.68,5936.12,2.628667
2,train,0,206,567.55,2.755097,84.480583,50.76,2952.66,9.459167
3,train,1,305,865.93,2.839115,87.000000,67.58,5059.96,14.432167
4,val,0,44,131.64,2.991818,91.659091,2952.66,4598.23,2.194000
5,val,1,66,164.80,2.496970,76.848485,5059.96,5475.68,2.746667



Overall:


,split,clips,total_duration_sec,speakers,total_duration_min
0,test,111,253.41,2,4.223500
1,train,511,1433.48,2,23.891333
2,val,110,296.44,2,4.940667


## Cell 7 — Leakage Check

เช็กว่า clip เดียวกันไม่อยู่หลาย split

In [7]:
clip_split_counts = (
    split_df
    .groupby("clip_id")["split"]
    .nunique()
)

leaked_clips = (
    clip_split_counts
    [clip_split_counts > 1]
)

print(
    "Clips appearing in >1 split:",
    len(leaked_clips)
)

if len(leaked_clips) > 0:
    raise RuntimeError(
        "พบ clip leakage"
    )

print("Clip leakage check: PASS")

Clips appearing in >1 split: 0
Clip leakage check: PASS


## Cell 8 — Temporal Boundary Check

แสดงขอบเขตเวลาแต่ละ speaker เพื่อให้เห็นว่า split ไม่สุ่มปนกัน

In [8]:
for speaker_id in sorted(
    split_df["speaker_id"]
    .unique()
):

    print(
        f"\nSpeaker {speaker_id}"
    )

    s = split_df[
        split_df["speaker_id"]
        == speaker_id
    ]

    for split_name in [
        "train",
        "val",
        "test",
    ]:

        part = s[
            s["split"]
            == split_name
        ]

        if len(part) == 0:
            print(
                split_name,
                "EMPTY"
            )
            continue

        print(
            f"{split_name:5s} | "
            f"clips={len(part):3d} | "
            f"time={part['start_time'].min():.2f}"
            f" → {part['end_time'].max():.2f}"
        )


Speaker 0
train | clips=206 | time=50.76 → 2952.66
val   | clips= 44 | time=2952.66 → 4598.23
test  | clips= 45 | time=4598.23 → 5905.77

Speaker 1
train | clips=305 | time=67.58 → 5059.96
val   | clips= 66 | time=5059.96 → 5475.68
test  | clips= 66 | time=5475.68 → 5936.12


## Cell 9 — Text Statistics per Split

ดูความยาวข้อความและจำนวนอักษรโดยรวม

In [9]:
split_df["text_length"] = (
    split_df["text"]
    .astype(str)
    .str.len()
)

text_summary = (
    split_df
    .groupby("split")
    .agg(
        clips=("clip_id", "count"),
        mean_text_length=("text_length", "mean"),
        median_text_length=("text_length", "median"),
        min_text_length=("text_length", "min"),
        max_text_length=("text_length", "max"),
    )
    .reset_index()
)

display(text_summary)

,split,clips,mean_text_length,median_text_length,min_text_length,max_text_length
0,test,111,30.117117,29.0,3,69
1,train,511,38.616438,36.0,3,115
2,val,110,37.336364,36.0,3,114


## Cell 10 — Character Coverage

สร้างชุดตัวอักษรจาก Train แล้วเช็กว่า Val/Test มีตัวอักษรที่ Train ไม่เคยเห็นหรือไม่

สำหรับ character-level baseline ข้อมูลนี้สำคัญมาก

In [10]:
def charset_from_texts(texts):
    chars = set()

    for text in texts:
        chars.update(
            list(
                str(text)
            )
        )

    return chars


train_chars = charset_from_texts(
    split_df[
        split_df["split"] == "train"
    ]["text"]
)

val_chars = charset_from_texts(
    split_df[
        split_df["split"] == "val"
    ]["text"]
)

test_chars = charset_from_texts(
    split_df[
        split_df["split"] == "test"
    ]["text"]
)

val_unseen = sorted(
    val_chars - train_chars
)

test_unseen = sorted(
    test_chars - train_chars
)

print("Train charset size:", len(train_chars))
print("Val unseen chars   :", val_unseen)
print("Test unseen chars  :", test_unseen)

Train charset size: 161
Val unseen chars   : ['д', 'л', 'н', '这', '는']
Test unseen chars  : ['*', '7', '9', 'ł']


## Cell 11 — Verify Frame Paths สำหรับแต่ละ Split

สุ่มตรวจ path เพื่อไม่เสียเวลาตรวจทุกไฟล์ซ้ำ

In [11]:
from pathlib import Path
import random

random.seed(42)

for split_name in [
    "train",
    "val",
    "test",
]:

    part = split_df[
        split_df["split"]
        == split_name
    ]

    sample_n = min(
        20,
        len(part)
    )

    sample_rows = (
        part.sample(
            n=sample_n,
            random_state=42
        )
        if sample_n > 0
        else part
    )

    missing = 0
    checked = 0

    for _, row in sample_rows.iterrows():

        paths = str(
            row["frame_paths"]
        ).split("|")

        # เช็กต้น กลาง ท้าย
        if not paths:
            continue

        check_indices = sorted(
            set(
                [
                    0,
                    len(paths) // 2,
                    len(paths) - 1,
                ]
            )
        )

        for idx in check_indices:

            p = Path(
                paths[idx]
            )

            full = (
                p
                if p.is_absolute()
                else PROJECT_ROOT / p
            )

            checked += 1

            if not full.exists():
                missing += 1

    print(
        f"{split_name}: "
        f"checked={checked}, "
        f"missing={missing}"
    )

train: checked=60, missing=0
val: checked=60, missing=0
test: checked=60, missing=0


## Cell 12 — Export Split CSVs

In [12]:
TRAIN_CSV = (
    OUTPUT_ROOT
    / "train.csv"
)

VAL_CSV = (
    OUTPUT_ROOT
    / "val.csv"
)

TEST_CSV = (
    OUTPUT_ROOT
    / "test.csv"
)

ALL_SPLIT_CSV = (
    OUTPUT_ROOT
    / "all_splits.csv"
)

SUMMARY_CSV = (
    OUTPUT_ROOT
    / "split_summary.csv"
)

split_df[
    split_df["split"]
    == "train"
].to_csv(
    TRAIN_CSV,
    index=False,
    encoding="utf-8-sig"
)

split_df[
    split_df["split"]
    == "val"
].to_csv(
    VAL_CSV,
    index=False,
    encoding="utf-8-sig"
)

split_df[
    split_df["split"]
    == "test"
].to_csv(
    TEST_CSV,
    index=False,
    encoding="utf-8-sig"
)

split_df.to_csv(
    ALL_SPLIT_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(" -", TRAIN_CSV)
print(" -", VAL_CSV)
print(" -", TEST_CSV)
print(" -", ALL_SPLIT_CSV)
print(" -", SUMMARY_CSV)

Saved:
 - D:\Face_detection\dataset\splits\video001\train.csv
 - D:\Face_detection\dataset\splits\video001\val.csv
 - D:\Face_detection\dataset\splits\video001\test.csv
 - D:\Face_detection\dataset\splits\video001\all_splits.csv
 - D:\Face_detection\dataset\splits\video001\split_summary.csv


## Cell 13 — Final Check

ถ้าผ่าน Cell นี้ ถือว่า Dataset พร้อมสำหรับ Baseline Training

In [13]:
train_df = pd.read_csv(
    OUTPUT_ROOT / "train.csv"
)

val_df = pd.read_csv(
    OUTPUT_ROOT / "val.csv"
)

test_df = pd.read_csv(
    OUTPUT_ROOT / "test.csv"
)

print("=" * 70)
print("FINAL SPLIT")
print("=" * 70)

print("Train clips:", len(train_df))
print("Val clips  :", len(val_df))
print("Test clips :", len(test_df))

print(
    "Total:",
    len(train_df)
    + len(val_df)
    + len(test_df)
)

print("\nSpeakers per split:")

for name, part in [
    ("Train", train_df),
    ("Val", val_df),
    ("Test", test_df),
]:
    print(
        name,
        sorted(
            part["speaker_id"]
            .unique()
            .tolist()
        )
    )

assert (
    len(train_df)
    + len(val_df)
    + len(test_df)
    == len(df)
)

print("\nDataset split: PASS")

FINAL SPLIT
Train clips: 511
Val clips  : 110
Test clips : 111
Total: 732

Speakers per split:
Train [0, 1]
Val [0, 1]
Test [0, 1]

Dataset split: PASS


# ขั้นต่อไป — 08 Baseline Lip Reading Training 🔥

เมื่อ Cell 13 ขึ้น:

`Dataset split: PASS`

เราจะสร้าง baseline model:

```text
Mouth Frames 96×96
    ↓
Visual Encoder
    ↓
Temporal Encoder
    ↓
CTC
    ↓
Thai Character Sequence
```

Training รอบแรกมีเป้าหมายเพื่อ:
1. เช็กว่า loss ลด
2. เช็กว่า model overfit subset เล็กได้
3. ตรวจ CER/WER
4. ตรวจว่า frame/text alignment ใช้งานได้
5. เตรียม architecture สำหรับ dataset ที่ใหญ่ขึ้น

> ด้วยข้อมูลปัจจุบัน 2 speakers / ~33 นาที ผลยังไม่ใช่โมเดล production แต่เพียงพอสำหรับ baseline end-to-end experiment